In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
df = pd.read_csv('PPG_Features_Per_Heartbeat.csv')

y_SBP = df['SBP']
y_DBP = df['DBP']

groups = df['data_ID']

exclude_cols = [
    "data_ID", "struct", "SBP", "DBP", 
    "left_margin", "right_margin",
    "mean_sbp", "mean_dbp", 
    "sbp_class", "dbp_class"
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(gss.split(X,y_SBP,groups=groups))
X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_SBP_train = y_SBP.iloc[train_idx]
y_SBP_val = y_SBP.iloc[val_idx]

y_DBP_train = y_DBP.iloc[train_idx]
y_DBP_val = y_DBP.iloc[val_idx]


RF_SBP = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

RF_DBP = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# baseline error: the error of the model trained on all features
RF_SBP.fit(X_train,y_SBP_train)
baseline_error_SBP = mean_absolute_error(y_SBP_val, RF_SBP.predict(X_val))
print('Fitted RF_SBP')
RF_DBP.fit(X_train,y_DBP_train)
baseline_error_DBP = mean_absolute_error(y_DBP_val, RF_DBP.predict(X_val))
print('Fitted RF_DBP')


feature_importances_SBP = {}
feature_importances_DBP = {}
i = 0
for f in X_train.columns:
    X_train_drop = X_train.drop(columns=[f])
    X_val_drop = X_val.drop(columns=[f])

    RF_SBP.fit(X_train_drop,y_SBP_train)
    RF_DBP.fit(X_train_drop,y_DBP_train)
    error_drop_SDP = mean_absolute_error(y_SBP_val, RF_SBP.predict(X_val_drop))
    error_drop_DBP = mean_absolute_error(y_DBP_val, RF_DBP.predict(X_val_drop))
    
    feature_importances_SBP[f] = error_drop_SDP - baseline_error_SBP
    feature_importances_DBP[f] = error_drop_DBP - baseline_error_DBP
    i+=1
    print(f'fitted both models for feature {f} with index {i}')


sbp_importance = pd.Series(feature_importances_SBP).sort_values(ascending=False)
dbp_importance = pd.Series(feature_importances_DBP).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sbp_importance.sort_values(ascending=True).plot.barh(ax=ax[0], color="skyblue")
ax[0].set_title("SBP Feature Importance (Drop-column)")
ax[0].set_xlabel("Increase in MAE")
ax[0].set_ylabel("Feature")

dbp_importance.sort_values(ascending=True).plot.barh(ax=ax[1], color="lightgreen")
ax[1].set_title("DBP Feature Importance (Drop-column)")
ax[1].set_xlabel("Increase in MAE")
ax[1].set_ylabel("Feature")

plt.tight_layout()
plt.savefig("feature_importance_drop_column_per_heartbeat.png", dpi=300)
plt.show()


KeyError: 'data ID'